In [1]:
import os
import re
import json

In [2]:
# def extract_diff_block(text):
#     if "```diff" in text:
#         # Extract content between ```diff and the next ```
#         pattern = r"```diff\n(.*?)```"
#         matches = re.findall(pattern, text, re.DOTALL)
#         if matches:
#             return 'diff\n' + matches[0]
#         else:
#             return text  # Return original text if no match
#     else:
#         return text

def extract_diff_block(text):
    # Case 1: Look for ```diff or ```git blocks
    code_block_pattern = r"```(?:diff|git)\s*\n(.*?)```"
    matches = re.findall(code_block_pattern, text, re.DOTALL)
    if matches:
        return matches[0]
    
    # Case 2: Look for standard diff marker (```diff, ```git, or just diff without code block)
    diff_marker_pattern = r"(?:```)?diff\s*\n(.*?)(?:```|$)"
    matches = re.findall(diff_marker_pattern, text, re.DOTALL)
    if matches:
        return matches[0]
    
    # Case 3: Look for traditional diff format with ---/+++ pattern
    traditional_diff_pattern = r"(---\s+[^\n]+\n\+\+\+\s+[^\n]+\n(?:@@[\s\-\+\d,]+@@.*\n)?(?:[\s\+\-\\].*\n)*)"
    matches = re.findall(traditional_diff_pattern, text, re.DOTALL)
    if matches:
        return matches[0]
    
    # Case 4: Look for any content with leading +/- characters that resembles diff
    diff_like_pattern = r"((?:^[\+\-].*\n)+)"
    matches = re.findall(diff_like_pattern, text, re.MULTILINE)
    if matches:
        return "\n".join(matches)
    
    # Default: return the original text if no diff-like content is found
    return text

In [3]:
def read_diff_files_to_json(folder_path, output_json):
    diff_data = []
    
    # Check if the given path is a directory
    if not os.path.isdir(folder_path):
        print(f"Error: {folder_path} is not a valid directory.")
        return
    
    # Iterate over files in the folder
    for file_name in os.listdir(folder_path):
        if file_name.endswith(".diff"):  # Check if the file is a .diff file
            file_path = os.path.join(folder_path, file_name)
            instance_id = file_name.replace('.diff', '').replace('_deepseek', '')
            
            with open(file_path, "r", encoding="utf-8") as file:
                diff_patch = extract_diff_block(file.read())

            diff_data.append(
                {
                    "instance_id": instance_id, 
                    "model_patch": diff_patch, 
                    "model_name_or_path": "chatgpt", 
                }
            )
                # diff_data[instance_id] = file.read()
    
    # Write the dictionary to a JSON file
    try:
        with open(output_json, "w", encoding="utf-8") as json_file:
            json.dump(diff_data, json_file)
        print(f"Successfully stored data in {output_json}")
    except Exception as e:
        print(f"Error writing to JSON file: {e}")


In [4]:
folder_path = "./test_outputs/chatgpt/ast_based"
output_json = f"{folder_path}/patches.json"
read_diff_files_to_json(folder_path, output_json)

Successfully stored data in ./test_outputs/chatgpt/ast_based/patches.json


In [ ]:
python3 -m swebench.harness.run_evaluation     --dataset_name princeton-nlp/SWE-bench_Lite     --predictions_path /home/tweichuan/project/test_outputs/chatgpt/ast_based/patches.json     --max_workers 8     --run_id gpt_ast_format

SyntaxError: invalid syntax (3809182126.py, line 1)

In [ ]:
output_json

'./test_outputs/chatgpt/direct_format/patches.json'